> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# Turtle breakout with ATR pyramiding: does it beat the CDI and the IBOV?

The rules are the owner's TradingView script (`turtle/turtle.pine`) as written in `docs/brief.md` section 7c: buy a close above the highest high of the previous 20 candles, add a unit every 1 ATR above the average price (up to 4 units), leave on a close below the lowest low of the previous 20 candles or on a close more than 1 ATR below the average price. Benchmarks are in `docs/cdi.md` and `docs/ibov.md`. The notebook holds no trading logic: it calls `b3core` and prints tables.

**Before trusting any number here:**
- **There is no capital cap (owner's decision, still an open question).** `capital_brl` only sets the risk budget. With a 3% ATR one unit costs about 17,000 and four units about 67,000 against 25,000 of capital. A CAGR on `capital_brl` therefore hides leverage. Every table shows **peak money in the position divided by capital** next to the return: read the two together.
- **Two fill rules.** `signal_close` is the script (every order fills at the signal candle's close, which is optimistic: the real close is an auction). `next_open` fills at the next open and is the realistic one. If they differ a lot, the strategy depends on the optimistic fill. Section 5 compares them.
- **The stop is read on the close**, not a resting order, so a gap can lose more than one ATR.
- **The fixed setting is the script's own** (20, 20, 14, 1 ATR, 4 units): no value was tuned on this data.
- **The ticker lists are drafts from memory, and "most traded" uses recent data.** That is survivorship bias. Tickers with no data are listed, never silently dropped.
- **BDRs carry the dollar**, and their Yahoo history is short and often flat. The IBOV and the CDI are not like-for-like for them.
- **Not modelled:** taxes, slippage beyond the 0.03% fee on every fill, dividends (raw closes), and a shared-cash rule for simultaneous signals. The portfolio is independent slots (`b3core.portfolio`).
- **The engine counts, and warns about, two things:** units skipped because they rounded below one lot, and candles that justified more than one add (only one is made). The tables below sum them instead of printing a warning per run.

In [ ]:
import json
import urllib.request
import warnings
from dataclasses import replace
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

from b3core.benchmark import compare_with_index
from b3core.portfolio import buy_and_hold_equity, combine_equity, summarize_portfolio
from b3core.turtle import TurtleConfig, run_turtle
from b3core.universe import scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data, shared with the IFR2 notebooks
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Parameters

`CONFIG` is the script's own setting plus the B3 fee of 0.03% on every fill (`TurtleConfig.preset`). Everything else is a list or a size you can change.

In [ ]:
CONFIG = TurtleConfig.preset()  # entry 20, exit 20, ATR 14, add every 1 ATR, 4 units, capital 25,000, risk 2%, fee 0.03%
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive
OOS_FRACTION = 0.3

# Draft candidates, from memory: edit freely. Tickers that fail to download are reported below.
IBOV_CANDIDATES = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
    "ITSA4", "KLBN11", "TAEE11", "SANB11", "BPAC11", "HAPV3", "COGN3", "BEEF3", "MRFG3", "CPLE6",
    "EGIE3", "SMTO3", "RDOR3", "HYPE3", "CCRO3", "UGPA3", "VBBR3", "EMBR3", "ALPA4", "MULT3",
    "BRFS3", "ASAI3", "GMAT3", "CMIN3", "ENEV3", "RRRP3",
]
BDR_CANDIDATES = [
    "AAPL34", "MSFT34", "AMZO34", "GOGL34", "NVDC34", "TSLA34", "M1TA34", "NFLX34",
    "DISB34", "COCA34", "JPMC34", "BERK34", "MUTC34", "ROXO34", "MELI34", "BABA34",
]
N_LIQUID_IBOV = 30  # the Ibovespa universe: the most traded of the candidates
N_LIQUID_BDR = 10  # the BDR universe: the ten most traded of the candidates
PICK_SIZES = [5, 8, 10]  # how many Ibovespa names to keep when choosing the best in-sample
MIN_CANDLES_IBOV = 750  # about three years of daily candles
MIN_CANDLES_BDR = 500  # BDRs are younger: about two years
LIQUIDITY_WINDOW = 250  # candles used to measure the median traded value

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the shared cache folder (the files `ifr2_universe.ipynb` already downloaded are reused). A **data-quality table** follows, because thin history and flat days matter most for BDRs.

In [ ]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


def load_ibov() -> pd.Series:
    path = CACHE / "ibov.csv"
    if not path.exists():
        raw = yf.download("^BVSP", start=START, end=END, auto_adjust=False, progress=False)
        raw.columns = raw.columns.get_level_values(0)
        raw[["Close"]].dropna().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)["Close"].dropna()


def local_day(candles: pd.DataFrame) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(candles["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()


ibov = load_ibov()


def load_universe(tickers: list[str], min_candles: int) -> tuple[dict, dict]:
    """Raw candles per ticker, and the tickers left out (no data, or too little history)."""
    loaded = {t: load_candles(t) for t in tickers}
    out = {
        "no data": [t for t, c in loaded.items() if c is None],
        "too short": [t for t, c in loaded.items() if c is not None and len(c) < min_candles],
    }
    data = {t: c for t, c in loaded.items() if c is not None and len(c) >= min_candles}
    return data, out


ibov_all, ibov_out = load_universe(IBOV_CANDIDATES, MIN_CANDLES_IBOV)
bdr_all, bdr_out = load_universe(BDR_CANDIDATES, MIN_CANDLES_BDR)
print(f"Ibovespa candidates: {len(ibov_all)} loaded; left out: {ibov_out}")
print(f"BDR candidates: {len(bdr_all)} loaded; left out: {bdr_out}")
print(f"IBOV: {len(ibov)} closes, {ibov.index[0].date()} to {ibov.index[-1].date()}")

In [ ]:
def quality(data: dict) -> pd.DataFrame:
    """Plain data checks: history, zero-volume days, flat days (high equals low), the longest gap."""
    rows = {}
    for t, c in data.items():
        days = local_day(c)
        rows[t] = {
            "candles": len(c), "first": days[0].date(), "last": days[-1].date(),
            "zero volume %": float((c["volume"] == 0).mean() * 100),
            "flat days %": float((c["high"] == c["low"]).mean() * 100),
            "longest gap (days)": int(pd.Series(days).diff().dt.days.max()),
        }
    return pd.DataFrame(rows).T


quality(bdr_all)

## 3. The CDI

The benchmark is the daily CDI compounded over each window (`docs/cdi.md`). The Central Bank API is tried first; if it is blocked, the **Selic target minus 0.10 points a year** is used, which is an approximation (the cell prints which one). A file `samples/ifr2_universe/cdi.csv` (series 12, columns `date` and `rate`) overrides both.

In [ ]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


trading_days = sorted(set().union(*[set(local_day(c)) for d in (ibov_all, bdr_all) for c in d.values()]))


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(trading_days).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}")

## 4. Clean the candles, then build the two universes

Two kinds of row cannot be compared with the CDI, and they are **dropped and printed**, never filled:
- **Candles before the first IBOV close** (`docs/ibov.md`): Yahoo's index series starts a day after the first stock candle.
- **Candles on days with no CDI rate.** With the exact CDI (series 12) these are holidays that Yahoo pads with a zero-volume row, and the days after the latest published rate. With the Selic-target fallback every weekday has a rate, so this check drops nothing.

**Most traded** is then the median of `close * volume` over the last `LIQUIDITY_WINDOW` candles of each ticker. It is measured on the most recent data, so it is a survivor ranking, not what was liquid in 2013.

In [ ]:
def clean(data: dict) -> tuple[dict, pd.DataFrame]:
    cleaned, rows = {}, {}
    cdi_days = set(pd.DatetimeIndex(cdi.index).normalize())
    for t, c in data.items():
        days = local_day(c)
        before_ibov = days < ibov.index[0]
        no_rate = ~before_ibov & ~days.isin(cdi_days) & (days > ibov.index[0])
        zero_volume = no_rate & (c["volume"].to_numpy() == 0)
        keep = ~before_ibov & ~no_rate
        cleaned[t] = c[keep].reset_index(drop=True)
        rows[t] = {
            "before first IBOV close": int(before_ibov.sum()),
            "no CDI rate": int(no_rate.sum()),
            "...of which zero volume": int(zero_volume.sum()),
        }
    return cleaned, pd.DataFrame(rows).T


def traded_value(data: dict) -> pd.Series:
    return pd.Series(
        {t: float((c["close"] * c["volume"]).tail(LIQUIDITY_WINDOW).median()) for t, c in data.items()}
    ).sort_values(ascending=False)


ibov_clean, ibov_dropped = clean(ibov_all)
bdr_clean, bdr_dropped = clean(bdr_all)
dropped = pd.concat([ibov_dropped, bdr_dropped])
print(f"{int((dropped > 0).any(axis=1).sum())} of {len(dropped)} tickers had rows dropped. Most dropped for one ticker:")
print(dropped.max().to_string())

ibov_liquidity = traded_value(ibov_clean)
bdr_liquidity = traded_value(bdr_clean)
ibov_data = {t: ibov_clean[t] for t in ibov_liquidity.index[:N_LIQUID_IBOV]}
bdr_data = {t: bdr_clean[t] for t in bdr_liquidity.index[:N_LIQUID_BDR]}
print(f"Ibovespa universe ({len(ibov_data)}): {sorted(ibov_data)}")
print(f"BDR universe ({len(bdr_data)}): {sorted(bdr_data)}")
pd.DataFrame({"ibov: median traded value (R$)": ibov_liquidity, "bdr: median traded value (R$)": bdr_liquidity})

## 5. One shared out-of-sample date per universe

Each universe is cut at the date that leaves roughly the last `OOS_FRACTION` of its trading days out. Everything that is *chosen* (which Ibovespa names to keep) is chosen on the in-sample part only.

In [ ]:
def split(data: dict) -> tuple[dict, dict, pd.Timestamp]:
    days = {t: local_day(c) for t, c in data.items()}
    all_days = sorted(set().union(*days.values()))
    cut = all_days[int(len(all_days) * (1 - OOS_FRACTION))]
    return (
        {t: c[days[t] < cut].reset_index(drop=True) for t, c in data.items()},
        {t: c[days[t] >= cut].reset_index(drop=True) for t, c in data.items()},
        cut,
    )


ibov_is, ibov_oos, IBOV_CUT = split(ibov_data)
bdr_is, bdr_oos, BDR_CUT = split(bdr_data)
print(f"Ibovespa universe: out-of-sample starts {IBOV_CUT.date()}")
print(f"BDR universe: out-of-sample starts {BDR_CUT.date()}")

## 3. Helpers: running Turtle over a universe

`run_all` runs the fixed setting on every ticker. The engine's warnings (skipped units, several adds due on one candle) would print once per run, so they are silenced here and **summed in the tables** instead. `exposure` shows what the strategy actually did: how many units, how much money, how often it was in a trade.

In [ ]:
def run_all(data: dict, config: TurtleConfig) -> dict:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # the counts are in the result and summed in the tables
        return {t: run_turtle(c, config) for t, c in data.items()}


def exposure(data: dict, config: TurtleConfig) -> pd.DataFrame:
    """What the strategy did on each ticker: trades, units, money in the position, time in trades."""
    rows = {}
    for t, r in run_all(data, config).items():
        trades = r.trades
        rows[t] = {
            "trades": len(trades),
            "avg units": float(np.mean([x.units for x in trades])) if trades else 0.0,
            "max units": max((x.units for x in trades), default=0),
            "% candles in a closed trade": 100 * sum(x.days for x in trades) / len(data[t]),
            "peak money / capital": r.peak_cost / config.capital_brl,
            "skipped units": r.skipped_units,
            "multi-add candles": r.multi_add_candles,
            "open units at end": r.open_units,
        }
    return pd.DataFrame(rows).T


SCAN_COLUMNS = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1",
                "bh_cagr", "beats_bh", "ibov_cagr", "beats_ibov"]


def scan_one(data: dict, config: TurtleConfig, min_candles: int) -> pd.DataFrame:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return scan_universe(data, config, cdi, min_candles=min_candles, index_closes=ibov, runner=run_turtle)


def scan_both(in_sample: dict, out_of_sample: dict, config: TurtleConfig, min_is: int, min_oos: int) -> pd.DataFrame:
    scan_is = scan_one(in_sample, config, min_is)
    scan_oos = scan_one(out_of_sample, config, min_oos)
    print("skipped in-sample:", scan_is.attrs["skipped"] or "none", "| out-of-sample:", scan_oos.attrs["skipped"] or "none")
    table = scan_is[SCAN_COLUMNS].add_prefix("is_").join(scan_oos[SCAN_COLUMNS].add_prefix("oos_"), how="left")
    return table.sort_values("is_score_v1", ascending=False)

## 4. Ibovespa names: the script's setting on every ticker

The fixed `CONFIG` on each ticker, in-sample and out-of-sample side by side. `excess` is the strategy's CAGR minus the CDI's over the same window, `beats_cdi` is true when it is positive. Read `max_equity_drawdown` and `calmar` next to it, and the exposure table below for the leverage hidden behind the CAGR.

In [ ]:
scan_ibov = scan_both(ibov_is, ibov_oos, CONFIG, MIN_CANDLES_IBOV // 2, 120)
scan_ibov

In [ ]:
both = scan_ibov.dropna(subset=["oos_beats_cdi"])
pd.Series(
    {
        "tickers": len(scan_ibov),
        "beat the CDI in-sample": int(scan_ibov["is_beats_cdi"].sum()),
        "beat the CDI out-of-sample": int(both["oos_beats_cdi"].sum()),
        "beat it in both": int((both["is_beats_cdi"] & both["oos_beats_cdi"]).sum()),
        "beat the IBOV out-of-sample": int(both["oos_beats_ibov"].sum()),
        "median in-sample excess": scan_ibov["is_excess"].median(),
        "median out-of-sample excess": both["oos_excess"].median(),
    },
    name="summary",
).to_frame()

**What the strategy did, full history.** `peak money / capital` is the most money the open units ever held divided by `capital_brl`: 1 means the whole capital, 3 means three times it (leverage the spec does not allow yet). `% candles in a closed trade` is the share of candles inside a position that has closed.

In [ ]:
exposure(ibov_data, CONFIG).sort_values("peak money / capital", ascending=False)

## 5. The two fill rules, side by side

`signal_close` is the script. `next_open` signals on the close and fills at the next open, with the quantity and the levels of the signal candle. The same setting, the same tickers, the same windows. `beat` columns count tickers. A big gap between the rows means the result depends on the optimistic close fill.

In [ ]:
def compare_settings(settings: dict, in_sample: dict, out_of_sample: dict, min_is: int, min_oos: int) -> pd.DataFrame:
    rows = []
    for label, config in settings.items():
        for window, data, min_candles in (("in-sample", in_sample, min_is), ("out-of-sample", out_of_sample, min_oos)):
            table = scan_one(data, config, min_candles)
            results = {t: r for t, r in run_all(data, config).items() if t in table.index}
            peaks = pd.Series({t: r.peak_cost / config.capital_brl for t, r in results.items()})
            rows.append({
                "setting": label, "window": window, "tickers": len(table),
                "median cagr": table["cagr"].median(), "median excess": table["excess"].median(),
                "beat CDI": int(table["beats_cdi"].sum()), "beat B&H": int(table["beats_bh"].sum()),
                "beat IBOV": int(table["beats_ibov"].sum()),
                "median equity dd": table["max_equity_drawdown"].median(),
                "median peak money / capital": peaks.median(), "max peak money / capital": peaks.max(),
                "skipped units": sum(r.skipped_units for r in results.values()),
                "multi-add candles": sum(r.multi_add_candles for r in results.values()),
            })
    return pd.DataFrame(rows).set_index(["setting", "window"])


FILLS = {
    "signal_close (script)": CONFIG,
    "next_open": replace(CONFIG, fill_at="next_open"),
}
fill_study = compare_settings(FILLS, ibov_is, ibov_oos, MIN_CANDLES_IBOV // 2, 120)
fill_study

## 6. Does pyramiding help?

`max_units` 1 is the same breakout and exit with no adds, so the difference to 4 is what the pyramiding adds (return, drawdown and money in the position). Same fill rule as the script.

In [ ]:
UNITS = {f"max_units {n}": replace(CONFIG, max_units=n) for n in (1, 2, 4)}
units_study = compare_settings(UNITS, ibov_is, ibov_oos, MIN_CANDLES_IBOV // 2, 120)
units_study

## 7. A portfolio of independent slots

One ticker leaves most of the capital idle, so the capital is split equally between the tickers (`b3core.portfolio`, independent slots of `capital / N`, no shared-cash rule). Each slot keeps `risk_pct` of **its own** capital, so the portfolio's risk per unit shrinks as N grows, and there is still no cap on the money in a slot. `sum of slot peaks / capital` adds the peak of every slot: an upper bound for the money the portfolio could hold at once, because the peaks need not coincide.

In [ ]:
CAPITAL = CONFIG.capital_brl


def slot_curves(data: dict, n: int, config: TurtleConfig) -> tuple[dict, dict]:
    """The equity curve and the run of each ticker in a slot of CAPITAL / n."""
    slot = replace(config, capital_brl=CAPITAL / n)
    runs = run_all(data, slot)
    curves = {t: pd.Series(r.equity, index=pd.DatetimeIndex(data[t]["ts"])) for t, r in runs.items()}
    return curves, runs


def portfolio_of(curves: dict, runs: dict, members) -> dict:
    equity = combine_equity({t: curves[t] for t in members})
    summary = summarize_portfolio(equity, CAPITAL, cdi)
    held_index = compare_with_index(summary.cagr, summary.years, ibov, equity.index)
    return {
        **summary.__dict__, "ibov_cagr": held_index.cagr, "beats_ibov": held_index.beats,
        "sum of slot peaks / capital": sum(runs[t].peak_cost for t in members) / CAPITAL,
    }


def held_of(data: dict, members) -> dict:
    held = summarize_portfolio(buy_and_hold_equity({t: data[t] for t in members}, CAPITAL), CAPITAL, cdi)
    return {"bh_cagr": held.cagr, "bh_max_equity_drawdown": held.max_equity_drawdown}


def all_tickers_portfolio(universe: str, in_sample: dict, out_of_sample: dict) -> pd.DataFrame:
    rows = []
    for fill, config in FILLS.items():
        for window, data in (("in-sample", in_sample), ("out-of-sample", out_of_sample)):
            names = sorted(data)
            curves, runs = slot_curves(data, len(names), config)
            result = portfolio_of(curves, runs, names)
            held = held_of(data, names)
            rows.append({
                "universe": universe, "fill": fill, "window": window, "tickers": len(names),
                "cagr": result["cagr"], "equity dd": result["max_equity_drawdown"], "calmar": result["calmar"],
                "excess vs CDI": result["excess"], "beats CDI": result["beats_cdi"],
                "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
                "B&H cagr": held["bh_cagr"], "beats B&H": result["cagr"] > held["bh_cagr"],
                "sum of slot peaks / capital": result["sum of slot peaks / capital"],
            })
    return pd.DataFrame(rows).set_index(["universe", "fill", "window"])


all_tickers_portfolio("Ibovespa", ibov_is, ibov_oos)

### 7b. Keep the best N in-sample, judge them out-of-sample

The tickers are ranked by their in-sample `score_v1`, the best N form an equal-split portfolio, and it is judged on the out-of-sample part. The **worst N** are the control. `beats % of random` is the share of random N-ticker portfolios from the same universe with a lower out-of-sample CAGR: near 50% means the choice is no better than chance. Both fill rules.

In [ ]:
DRAWS = 300  # random subsets per size
rng = np.random.default_rng(0)  # seeded: the same subsets on every run
names = sorted(set(ibov_is) & set(ibov_oos))
by_score = [t for t in scan_ibov["is_score_v1"].dropna().sort_values(ascending=False).index if t in names]

rows = []
for n in PICK_SIZES:
    random_sets = [tuple(rng.choice(names, size=n, replace=False)) for _ in range(DRAWS)]
    for fill, config in FILLS.items():
        curves, runs = slot_curves({t: ibov_oos[t] for t in names}, n, config)
        random_cagr = pd.Series([portfolio_of(curves, runs, s)["cagr"] for s in random_sets])
        for label, members in (("best in-sample", by_score[:n]), ("worst in-sample (control)", by_score[-n:])):
            result = portfolio_of(curves, runs, members)
            held = held_of(ibov_oos, members)
            rows.append({
                "selection": label, "tickers": n, "fill": fill,
                "oos cagr": result["cagr"], "oos equity dd": result["max_equity_drawdown"],
                "oos calmar": result["calmar"], "oos excess": result["excess"], "beats CDI": result["beats_cdi"],
                "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
                "B&H cagr": held["bh_cagr"], "beats B&H": result["cagr"] > held["bh_cagr"],
                "sum of slot peaks / capital": result["sum of slot peaks / capital"],
                "beats % of random": float((random_cagr < result["cagr"]).mean()),
            })
print("best", PICK_SIZES[0], "in-sample:", by_score[: PICK_SIZES[0]])
pd.DataFrame(rows).set_index(["selection", "tickers", "fill"]).sort_index()

**How to read it.** A ranking that means something puts `best` above `worst` and above 50% of random. Then ask whether the best row beats the CDI *and* the IBOV with a drawdown you could live with, and whether `sum of slot peaks / capital` says the money needed is a multiple of the capital.

## 8. The ten most traded BDRs

No selection: the universe is already the ten most traded candidates. The same fixed setting on each BDR, then the ten as one equal-split portfolio, both fill rules.

In [ ]:
scan_bdr = scan_both(bdr_is, bdr_oos, CONFIG, MIN_CANDLES_BDR // 2, 100)
scan_bdr

In [ ]:
all_tickers_portfolio("BDR", bdr_is, bdr_oos)

**Caveats for the BDRs.** Ten correlated US mega-caps, a short and dollar-driven history, and an out-of-sample window of a few years: this is an anecdote, not evidence. The IBOV comparison is not like-for-like, and there is no US benchmark here.

## 9. What this notebook does not answer (yet)

- **A realistic account.** The capital cap is a deferred rule (`docs/brief.md` section 10). Until it exists, a Turtle CAGR on `capital_brl` should be read together with the money in the position, and a result that needs several times the capital is not a result an account of that size could obtain.
- **Is the part-time exposure judged fairly?** The CDI and the IBOV are fully invested every day, the strategy is not. A comparison over only the days in a trade, or a rule for what idle cash earns, needs an approved spec in `docs/` before any code.
- **Which signal wins when several fire together?** The independent slots have no shared cash, so a real account could not take every signal in a sell-off. That rule is a spec decision.
- **Parameter sensitivity.** Only the script's setting and `max_units` are tried. A grid over the channel lengths would need the optimizer to learn the Turtle strategy first, with the usual out-of-sample discipline.

Write what you conclude from the tables above here, in your own words, before relying on any of them.